In [ ]:
from google.cloud import bigquery

PROJECT_ID    = "qwiklabs-gcp-00-c521a9ba0b6e"
DATASET_ID    = "weather_comms"
RAW_TABLE     = "weather_data"
REPORT_TABLE  = "weather_data_with_reports"
CONNECTION_ID = "gemini_conn"
MODEL_NAME    = "gemini_model"
REGION        = "US"

client = bigquery.Client(project=PROJECT_ID)
print("Client ready.")

Client ready.


In [ ]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = REGION
client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset `{DATASET_ID}` ready.")

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_uri(
    "gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv",
    f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}",
    job_config=job_config,
)
load_job.result()

table = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}")
print(f"Loaded {table.num_rows:,} rows into `{RAW_TABLE}`.")

Dataset `weather_comms` ready.
Loaded 300 rows into `weather_data`.


In [ ]:
table = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}")
print("Columns:")
for field in table.schema:
    print(f"  {field.name:30s} {field.field_type}")

Columns:
  date                           DATE
  city                           STRING
  state                          STRING
  temperature_f                  FLOAT
  wind_speed_mph                 FLOAT
  precipitation_in               FLOAT
  barometric_pressure_inHg       FLOAT
  humidity_percent               FLOAT
  weather_condition              STRING


In [ ]:
client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}` LIMIT 10
""").to_dataframe()

,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition
0,2025-02-21,Atlanta,GA,55.7,5.0,0.12,29.80,50.4,Cloudy
1,2025-02-26,Atlanta,GA,75.2,10.4,0.03,29.58,49.9,Cloudy
2,2025-03-01,Atlanta,GA,51.7,4.7,0.08,29.74,49.9,Cloudy
3,2025-03-05,Atlanta,GA,74.4,5.1,0.02,29.92,50.4,Cloudy
4,2025-03-10,Atlanta,GA,59.5,9.6,0.09,29.67,57.2,Cloudy
5,2025-03-14,Atlanta,GA,71.7,7.2,0.18,29.92,55.3,Cloudy
6,2025-02-19,Boston,MA,61.7,3.9,0.11,29.62,54.1,Cloudy
7,2025-03-09,Boston,MA,76.7,4.3,0.09,29.52,40.9,Cloudy
8,2025-03-13,Boston,MA,71.9,9.8,0.16,29.99,42.3,Cloudy
9,2025-03-19,Boston,MA,60.7,6.4,0.04,29.83,49.4,Cloudy


In [ ]:
import subprocess, json

!bq mk --connection --location={REGION} --project_id={PROJECT_ID} \
    --connection_type=CLOUD_RESOURCE {CONNECTION_ID} || echo "Connection may already exist — continuing."

conn_info = subprocess.run(
    ["bq", "show", "--format=json", "--connection",
     f"{PROJECT_ID}.{REGION}.{CONNECTION_ID}"],
    capture_output=True, text=True
)
service_account = json.loads(conn_info.stdout)["cloudResource"]["serviceAccountId"]
print("Connection service account:", service_account)

Connection 931555514205.us.gemini_conn successfully created
Connection service account: bqcx-931555514205-cmy1@gcp-sa-bigquery-condel.iam.gserviceaccount.com


In [ ]:
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{service_account}" \
    --role="roles/aiplatform.user" \
    --condition=None

print("IAM role granted. Waiting ~60s for the permission to propagate...")
import time; time.sleep(60)

Updated IAM policy for project [qwiklabs-gcp-00-c521a9ba0b6e].
bindings:
- members:
  - serviceAccount:service-931555514205@gcp-sa-vertex-nb.iam.gserviceaccount.com
  role: roles/aiplatform.colabServiceAgent
- members:
  - serviceAccount:service-931555514205@gcp-sa-aiplatform-vm.iam.gserviceaccount.com
  role: roles/aiplatform.notebookServiceAgent
- members:
  - serviceAccount:service-931555514205@gcp-sa-aiplatform.iam.gserviceaccount.com
  role: roles/aiplatform.serviceAgent
- members:
  - serviceAccount:bqcx-931555514205-cmy1@gcp-sa-bigquery-condel.iam.gserviceaccount.com
  role: roles/aiplatform.user
- members:
  - serviceAccount:qwiklabs-gcp-00-c521a9ba0b6e@qwiklabs-gcp-00-c521a9ba0b6e.iam.gserviceaccount.com
  role: roles/bigquery.admin
- members:
  - serviceAccount:931555514205@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - serviceAccount:service-931555514205@gcp-sa-cloudbuild.iam.gserviceaccount.com
  role: roles/cloudbuild.serviceAgent
- m

In [ ]:
create_model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_NAME}`
REMOTE WITH CONNECTION `{PROJECT_ID}.{REGION}.{CONNECTION_ID}`
OPTIONS (ENDPOINT = 'gemini-2.5-flash')
"""

client.query(create_model_sql).result()
print(f"Remote model `{MODEL_NAME}` created, pointing to gemini-2.5-flash.")

Remote model `gemini_model` created, pointing to gemini-2.5-flash.


In [ ]:
generate_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{REPORT_TABLE}` AS
SELECT
    * EXCEPT(prompt, ml_generate_text_rai_result, ml_generate_text_status),
    ml_generate_text_llm_result AS weather_report
FROM ML.GENERATE_TEXT(
    MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_NAME}`,
    (
        SELECT
            *,
            CONCAT(
                'You are a public safety communications officer. ',
                'Write a short, clear weather report or warning for the public ',
                'based on the following conditions. Keep it under 60 words. ',
                'City: ', city, ', ', state,
                '. Date: ', CAST(date AS STRING),
                '. Temperature: ', CAST(temperature_f AS STRING), ' F',
                '. Condition: ', weather_condition,
                '. Wind speed: ', CAST(wind_speed_mph AS STRING), ' mph',
                '. Precipitation: ', CAST(precipitation_in AS STRING), ' in',
                '. Humidity: ', CAST(humidity_percent AS STRING), '%',
                '. Barometric pressure: ', CAST(barometric_pressure_inHg AS STRING), ' inHg'
            ) AS prompt
        FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}`
    ),
    STRUCT(
        0.3   AS temperature,          -- low = consistent, factual tone
        1024  AS max_output_tokens,
        TRUE  AS flatten_json_output   -- gives the clean `ml_generate_text_llm_result` column
    )
)
"""

client.query(generate_sql).result()
print(f"Weather reports generated and written to `{REPORT_TABLE}`.")

Weather reports generated and written to `weather_data_with_reports`.


In [ ]:
client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{REPORT_TABLE}` LIMIT 10
""").to_dataframe()

,ml_generate_text_llm_result,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition,weather_report
0,Atlanta: March 1st. Cloudy with light rain. Te...,2025-03-01,Atlanta,GA,51.7,4.7,0.08,29.74,49.9,Cloudy,Atlanta: March 1st. Cloudy with light rain. Te...
1,Atlanta: Cloudy with light rain today. Tempera...,2025-02-26,Atlanta,GA,75.2,10.4,0.03,29.58,49.9,Cloudy,Atlanta: Cloudy with light rain today. Tempera...
2,"Atlanta, today, February 21st: Cloudy with lig...",2025-02-21,Atlanta,GA,55.7,5.0,0.12,29.80,50.4,Cloudy,"Atlanta, today, February 21st: Cloudy with lig..."
3,Atlanta: Cloudy with light rain today. Tempera...,2025-03-10,Atlanta,GA,59.5,9.6,0.09,29.67,57.2,Cloudy,Atlanta: Cloudy with light rain today. Tempera...
4,Atlanta: Cloudy with light rain possible today...,2025-03-05,Atlanta,GA,74.4,5.1,0.02,29.92,50.4,Cloudy,Atlanta: Cloudy with light rain possible today...
5,"Atlanta: Friday, March 14th. Cloudy with light...",2025-03-14,Atlanta,GA,71.7,7.2,0.18,29.92,55.3,Cloudy,"Atlanta: Friday, March 14th. Cloudy with light..."
6,**Boston Weather Alert:** Cloudy with light ra...,2025-03-09,Boston,MA,76.7,4.3,0.09,29.52,40.9,Cloudy,**Boston Weather Alert:** Cloudy with light ra...
7,"Boston, today, February 19th: Cloudy with ligh...",2025-02-19,Boston,MA,61.7,3.9,0.11,29.62,54.1,Cloudy,"Boston, today, February 19th: Cloudy with ligh..."
8,"Boston, today, March 19th: Cloudy with light r...",2025-03-19,Boston,MA,60.7,6.4,0.04,29.83,49.4,Cloudy,"Boston, today, March 19th: Cloudy with light r..."
9,"Boston, today, March 13th: Expect cloudy skies...",2025-03-13,Boston,MA,71.9,9.8,0.16,29.99,42.3,Cloudy,"Boston, today, March 13th: Expect cloudy skies..."
